# Session 2: temperatures, work and the best pressure ratio
**A. Real-world problem.** A power-station turbine takes in 20 kg of air every second at
288 K. The metal of the turbine can survive gas at 1400 K. What are the temperatures at
each point of the engine, how much power comes out, and which pressure ratio gives the most?

Educational model: ideal Brayton cycle, air with constant properties, no losses, no fuel
mass. Not a design tool for real engines.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
GAMMA = 1.4        # ratio of specific heats (no unit)
CP = 1.005         # specific heat of air at constant pressure (kJ per kg per K)
K = (GAMMA - 1) / GAMMA

## Step 1: temperatures at the four states
State 1 is the inlet, 2 the compressor exit, 3 the turbine inlet (the hottest point) and
4 the turbine exit. Compression and expansion are ideal (no entropy change), so

    T2 = T1 x r^K        T4 = T3 / r^K

In [ ]:
def temperatures(t1, r, t3):
    """Return (T2, T4) in kelvin for inlet T1, pressure ratio r and turbine inlet T3."""
    if r < 1:
        raise ValueError("the pressure ratio must be at least 1")
    x = r ** K
    if t3 <= t1 * x:
        raise ValueError("T3 must be hotter than T2, otherwise the combustor cools the air")
    return t1 * x, t3 / x

## Step 2: work and heat per kilogram of air
With constant cp, each heat or work is cp times a temperature difference:

    w_c = cp (T2 - T1)        compressor work in
    w_t = cp (T3 - T4)        turbine work out
    w_net = w_t - w_c         net work
    q_in = cp (T3 - T2)       heat added in the combustor
    eta = w_net / q_in        (this equals 1 - 1 / r^K from Session 1)

In [ ]:
def cycle_work(t1, r, t3):
    """Return a dictionary of work and heat per kg of air (kJ/kg) and the efficiency."""
    t2, t4 = temperatures(t1, r, t3)
    w_c, w_t, q_in = CP * (t2 - t1), CP * (t3 - t4), CP * (t3 - t2)
    return {"t2": t2, "t4": t4, "w_c": w_c, "w_t": w_t, "w_net": w_t - w_c, "q_in": q_in, "eta": (w_t - w_c) / q_in}


def power_mw(mdot, t1, r, t3):
    """Net power in megawatts for an air flow mdot in kg/s (kJ/kg x kg/s = kW)."""
    return mdot * cycle_work(t1, r, t3)["w_net"] / 1000

## Step 3: the pressure ratio that gives the most work
Write x = r^K and tau = T3 / T1. Then w_net = cp T1 [tau (1 - 1/x) - (x - 1)]. Setting the
derivative with respect to x to zero gives x = sqrt(tau), so

    r_opt = (T3 / T1)^(1 / (2K))

Below r_opt the cycle is wasting hot gas, above it the compressor eats too much work.

In [ ]:
def best_ratio_for_work(t1, t3):
    return (t3 / t1) ** (1 / (2 * K))


def max_ratio(t1, t3):
    """Largest r for which T2 does not exceed T3 (the combustor would have nothing to do)."""
    return (t3 / t1) ** (1 / K)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t1, t3, mdot = 288, 1400, 20
    c = cycle_work(t1, 10, t3)
    print(f"r = 10: T2 = {c['t2']:.1f} K, T4 = {c['t4']:.1f} K")
    print(f"w_c = {c['w_c']:.1f}, w_t = {c['w_t']:.1f}, w_net = {c['w_net']:.1f} kJ/kg, eta = {c['eta']:.4f}")
    print(f"power for {mdot} kg/s = {power_mw(mdot, t1, 10, t3):.2f} MW")
    print(f"best r for work = {best_ratio_for_work(t1, t3):.1f}; largest possible r = {max_ratio(t1, t3):.0f}")

    r = np.linspace(2, 40, 190)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for t3x, col in ((1200, "tab:green"), (1400, "tab:blue"), (1600, "tab:red")):
        ax.plot(r, [cycle_work(t1, x, t3x)["w_net"] for x in r], color=col, label=f"T3 = {t3x} K")
        ro = best_ratio_for_work(t1, t3x)
        ax.plot(ro, cycle_work(t1, ro, t3x)["w_net"], "o", color=col)
    ax.set_xlabel("pressure ratio r (no unit)")
    ax.set_ylabel("net work (kJ per kg of air)")
    ax.set_title("Each curve peaks at its own best pressure ratio (dots)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "work_vs_ratio.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    ax2.plot(r, [cycle_work(t1, x, t3)["t2"] for x in r], color="tab:blue", label="T2 (compressor exit)")
    ax2.plot(r, [cycle_work(t1, x, t3)["t4"] for x in r], color="tab:red", label="T4 (turbine exit)")
    ax2.axhline(t3, color="gray", ls=":", label="T3 = 1400 K")
    ax2.set_xlabel("pressure ratio r (no unit)")
    ax2.set_ylabel("temperature (K)")
    ax2.set_title("Compressor exit gets hotter, turbine exit gets cooler")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "temperatures_vs_ratio.png", dpi=120)

## Where this stops
Real compressors and turbines do not reach the ideal temperatures: each needs more work
than the ideal one. Session 3 adds their efficiencies and uses what is left of the gas to
make thrust.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
"""Checkpoint for Session 2. Prints PASS or FAIL."""
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    c = L.cycle_work(288, 10, 1400)
    checks.append((f"worked example: T2 = 556.0 K (got {c['t2']:.1f})", abs(c["t2"] - 556.0) < 0.1))
    checks.append((f"worked example: T4 = 725.1 K (got {c['t4']:.1f})", abs(c["t4"] - 725.1) < 0.1))
    checks.append((f"worked example: w_c = 269.4 kJ/kg (got {c['w_c']:.1f})", abs(c["w_c"] - 269.4) < 0.1))
    checks.append((f"worked example: w_t = 678.2 kJ/kg (got {c['w_t']:.1f})", abs(c["w_t"] - 678.2) < 0.1))
    checks.append((f"worked example: w_net = 408.9 kJ/kg (got {c['w_net']:.1f})", abs(c["w_net"] - 408.9) < 0.1))
    checks.append((f"worked example: power for 20 kg/s is 8.18 MW (got {L.power_mw(20, 288, 10, 1400):.2f})",
                   abs(L.power_mw(20, 288, 10, 1400) - 8.18) < 0.01))
    checks.append(("consistency with Session 1: eta = 1 - 1 / r^K",
                   all(abs(L.cycle_work(288, r, 1400)["eta"] - (1 - 1 / r ** L.K)) < 1e-12 for r in (2, 5, 10, 25))))
    checks.append(("energy balance: w_net = q_in - q_out with q_out = cp (T4 - T1)",
                   abs(c["w_net"] - (c["q_in"] - L.CP * (c["t4"] - 288))) < 1e-9))
    ro = L.best_ratio_for_work(288, 1400)
    checks.append((f"design: the best pressure ratio for work is 15.9 (got {ro:.1f})", abs(ro - 15.91) < 0.01))
    w_best = L.cycle_work(288, ro, 1400)["w_net"]
    checks.append(("design: the work at r_opt beats the work at 0.8 r_opt and 1.25 r_opt",
                   w_best > L.cycle_work(288, 0.8 * ro, 1400)["w_net"] and w_best > L.cycle_work(288, 1.25 * ro, 1400)["w_net"]))
    checks.append(("a hotter turbine inlet gives more work at the same r",
                   L.cycle_work(288, 10, 1600)["w_net"] > L.cycle_work(288, 10, 1400)["w_net"] > L.cycle_work(288, 10, 1200)["w_net"]))
    checks.append(("edge case: r = 1 gives zero net work", abs(L.cycle_work(288, 1, 1400)["w_net"]) < 1e-9))
    for bad, label in ((lambda: L.cycle_work(288, 0.5, 1400), "r below 1"),
                       (lambda: L.cycle_work(288, 300, 1400), "T2 above T3 (r far too large)")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    return checks
import types
L = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
ok = True
for label, passed in run_checks(L):
    print(('PASS  ' if passed else 'FAIL  ') + label)
    ok &= bool(passed)
print('PASS' if ok else 'FAIL')